### Section 3.2 Parameter posterior geometry

In [1]:
import os, json, glob
import numpy as np
import xarray as xr
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

2026-11-25 10:18:34.429462: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-25 10:18:34.489479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795627114.530083   47908 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795627114.542010   47908 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-25 10:18:34.730788: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
from SALib.sample import saltelli
from SALib.analyze import sobol

In [3]:
# get config information
utils_dir = '../utils/'

with open(os.path.join(utils_dir, "biome_info.json"), "r") as f:
    biome_info = json.load(f)

with open(os.path.join(utils_dir, "param_info.json"), "r") as f:
    param_info = json.load(f)
universal_param_names = param_info['u_params']
pft_param_names = param_info['pft_params']

#### K-L divergence on Marginals Wave 4 NROY sets 

In [14]:
# Load NROY parameter sets
in_file = '../data/wave4_NROY_samples.csv'
NROY_samples = pd.read_csv(in_file,index_col=0)

In [16]:
samples = NROY_samples.loc[:, ~NROY_samples.columns.str.startswith("pft9_")]

In [18]:
N_BINS=20
ALPHA=0.05
N_NULL=1000
BIN_SWEEP = [5,8,10,15,20,30]
SEED=42

rng = np.random.default_rng(SEED)

In [19]:
def kl_from_uniform(x, n_bins, lo=0.0, hi=1.0):
    """
    D_KL(p || u) in nats for a single parameter, by histogram binning.
 
    With equal-width bins over the prior support, the uniform reference has
    probability 1/n_bins in every bin, so the divergence reduces to
    sum_k p_k * log(p_k * n_bins). Empty bins contribute zero.
    """
    counts, _ = np.histogram(x, bins=n_bins, range=(lo, hi))
    p = counts / counts.sum()
    nz = p > 0
    return float(np.sum(p[nz] * np.log(p[nz] * n_bins)))
 
 
def null_threshold(n_samples, n_bins, alpha=ALPHA, n_null=N_NULL, rng=rng):
    """
    Upper (1 - alpha) quantile of KL under the null that the marginal is
    uniform, for a sample of size n_samples binned into n_bins.
    """
    draws = rng.random((n_null, n_samples))
    null_kl = np.array([kl_from_uniform(d, n_bins) for d in draws])
    return float(np.quantile(null_kl, 1.0 - alpha)), null_kl

In [20]:
n_samples, n_params = samples.shape

names = samples.columns.values

thr, _ = null_threshold(n_samples, N_BINS)

kl = samples.apply(lambda c: kl_from_uniform(c.values, N_BINS))

res = pd.DataFrame({
    "parameter": kl.index,
    "kl_nats": kl.values,
    "threshold": thr,
    "constrained": kl.values > thr,
}).sort_values("kl_nats", ascending=False)

In [23]:
n_con = int(res["constrained"].sum())
print(f"n_samples = {n_samples}, n_params = {n_params}, n_bins = {N_BINS}")
print(f"null threshold (alpha = {ALPHA}) = {thr:.4f} nats")
print(f"constrained: {n_con} / {n_params}")
print(f"indistinguishable from prior: {n_params - n_con} / {n_params}")

n_samples = 1057, n_params = 236, n_bins = 20
null threshold (alpha = 0.05) = 0.0145 nats
constrained: 97 / 236
indistinguishable from prior: 139 / 236


In [24]:
N_BINS = 10
N_SAMPLED = 1_000_000_000
N_RETAINED = 1057

X = samples.values    
n_samp, n_par = X.shape
edges = np.linspace(0, 1, N_BINS + 1)

# fraction of NROY in each bin, per parameter
frac = np.zeros((n_par, N_BINS))
bidx = np.zeros(X.shape, dtype=int)
for j in range(n_par):
    b = np.clip(np.digitize(X[:, j], edges) - 1, 0, N_BINS - 1)
    bidx[:, j] = b
    frac[j] = np.bincount(b, minlength=N_BINS) / n_samp

# product of the marginal bin fractions at each NROY sample
log_p = np.log(frac[np.arange(n_par), bidx]).sum(axis=1)

# effective number of cells the independent product occupies
log_V_ind = -log_p.mean() - n_par * np.log(N_BINS)
V_joint = N_RETAINED / N_SAMPLED

ratio = np.exp(log_V_ind - np.log(V_joint))
print(f"V_joint = {V_joint:.3e}")
print(f"V_ind   = {np.exp(log_V_ind):.3e}")
print(f"ratio   = {ratio:.0f}x")

V_joint = 1.057e-06
V_ind   = 4.694e-04
ratio   = 444x


#### Parameter interactions analysis

In [4]:
# Config for the all-biome interaction screen
biomes = list(range(1, 14))          # biomes 1-13 have trained emulators (14-15 do not)
vars = ['gpp', 'lai', 'lh', 'tvc']
var_labels = {'gpp': 'GPP', 'lai': 'LAI', 'lh': 'LH', 'tvc': 'TVC'}

N = 512                              # Saltelli base sample size (matches the biome-2 analysis above)
S2_min = 0.01                        # significance floor: a pair must explain >= 1% of variance
num_resamples = 100                  # bootstrap resamples for the CIs
emulator_dir = '../clm-hmc/emulators'

In [5]:
def run_biome_sobol(b, N=N):
    """Run a second-order Sobol analysis for every variable in one biome.
    Returns (problem, {var: Si}).
    """
    param_names = biome_info[str(b)]['biome_parameters']
    problem = {
        'names': param_names,
        'num_vars': len(param_names),
        'bounds': [[0, 1]] * len(param_names),
    }

    param_values = saltelli.sample(problem, N, calc_second_order=True)

    results = {}
    for var in vars:
        emulator = tf.saved_model.load(f'{emulator_dir}/biome_{var}_{b}')
        Y, _ = emulator.compiled_predict_f(param_values)
        Y = Y.numpy().flatten()

        results[var] = sobol.analyze(
            problem, Y,
            calc_second_order=True,
            num_resamples=num_resamples,
            conf_level=0.95,
            print_to_console=False,
            seed=1,
        )
    return problem, results


def extract_significant_pairs(Si, problem, biome, var, S2_min=S2_min):
    """DataFrame of second-order pairs that are statistically resolved
    (95% CI excludes zero) and practically large (S2 >= S2_min).
    Returns an empty DataFrame if no pair qualifies.
    """
    names = problem['names']
    S2 = np.asarray(Si['S2'], float)
    S2c = np.asarray(Si['S2_conf'], float)
    n = len(names)

    rows = []
    for i in range(n):
        for j in range(i + 1, n):
            s2, conf = S2[i, j], S2c[i, j]
            if np.isnan(s2):
                continue
            ci_excludes_zero = (s2 - conf) > 0
            practically_large = s2 >= S2_min
            if ci_excludes_zero and practically_large:
                rows.append({
                    'biome': biome,
                    'biome_name': biome_info[str(biome)]['name'],
                    'variable': var_labels[var],
                    'param_i': names[i],
                    'param_j': names[j],
                    'S2': s2,
                    'S2_conf': conf,
                })
    return pd.DataFrame(rows)


def summarize_totals(Si, problem, biome, var):
    """Per biome/variable additivity summary: how much of the variance is main
    effects vs. interactions, plus the single largest total-interaction parameter.
    """
    s1 = np.asarray(Si['S1'], float)
    st = np.asarray(Si['ST'], float)
    interaction = np.clip(st - s1, 0, None)      # per-parameter total interaction effect
    top = int(np.argmax(interaction))

    return {
        'biome': biome,
        'biome_name': biome_info[str(biome)]['name'],
        'variable': var_labels[var],
        'S1_sum': s1.sum(),
        'ST_sum': st.sum(),
        'interaction_frac': float(np.clip(1 - s1.sum(), 0, 1)),   # fraction of variance from interactions
        'top_interacting_param': problem['names'][top],
        'top_param_interaction': interaction[top],
    }

In [ ]:
# Run the screen across all biomes and variables
all_pairs = []
all_summaries = []

for b in biomes:
    problem, results = run_biome_sobol(b)
    for var in vars:
        Si = results[var]
        all_pairs.append(extract_significant_pairs(Si, problem, b, var))
        all_summaries.append(summarize_totals(Si, problem, b, var))
    print(f"biome {b:2d} ({biome_info[str(b)]['name']}) done")

df_significant = (pd.concat(all_pairs, ignore_index=True)
                  if any(len(d) for d in all_pairs) else pd.DataFrame())
df_totals = pd.DataFrame(all_summaries)

2026-11-25 10:18:48.107358: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


biome  1 (Tropical rainforest) done
biome  2 (Tropical savanna) done
biome  3 (Subtropical savanna) done
biome  4 (Grasslands) done
biome  5 (Shrublands) done


In [ ]:
# Additivity summary: fraction of variance from interactions, biome x variable
pivot_frac = df_totals.pivot(index='biome_name', columns='variable', values='interaction_frac')
pivot_frac = pivot_frac.reindex([biome_info[str(b)]['name'] for b in biomes])[['GPP', 'LAI', 'LH', 'TVC']]

print("Interaction fraction (1 - sum S1), by biome and variable:")
display(pivot_frac.round(3))
print(f"\nrange: {df_totals.interaction_frac.min():.3f} - {df_totals.interaction_frac.max():.3f} "
      f"(mean {df_totals.interaction_frac.mean():.3f})")

# Significant pairwise interactions
print(f"\nSignificant interaction pairs (CI excludes 0 and S2 >= {S2_min}): {len(df_significant)}")
if len(df_significant):
    display(df_significant.sort_values('S2', ascending=False).reset_index(drop=True))


In [ ]:
# Heatmap: fraction of emulated variance from interactions, biome x variable
fig, ax = plt.subplots(figsize=(4.5, 6))
sns.heatmap(
    pivot_frac, annot=True, fmt='.2f', cmap=cmap,
    vmin=0, vmax=max(0.25, np.nanmax(pivot_frac.values)),
    cbar_kws={'label': 'interaction fraction  $1 - \\sum S_1$', 'shrink': 0.6},
    linewidths=0.4, linecolor='white', ax=ax,
)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('Emulated variance from parameter interactions', fontsize=10, pad=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('../figs/SI_FigS12_sobol_interactions.png', dpi=300, bbox_inches='tight')